# Akuisisi Data Time-Series JISDOR USD/IDR

Menyiapkan data harian USD/IDR dari **Bank Indonesia (JISDOR)** untuk periode **1 September 2021 – 1 September 2026**. Hasil akhir diekspor ke `data/processed/jisdor_usd_idr.csv`.

## 1. Import & Konfigurasi

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

START_DATE = pd.Timestamp("2021-09-01")
END_DATE = pd.Timestamp("2026-09-01")
RAW_FILENAME = "Informasi_Kurs_Jisdor.xlsx"
SHEET_NAME = "Informasi Kurs Jisdor"


def find_repo_root(start: Path) -> Path:
    """Cari root repo berdasarkan lokasi file raw di data/raw/."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / RAW_FILENAME).exists():
            return candidate
    raise FileNotFoundError(f"{RAW_FILENAME} tidak ditemukan di data/raw/.")


ROOT_DIR = find_repo_root(Path.cwd())
RAW_PATH = ROOT_DIR / "data" / "raw" / RAW_FILENAME
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "jisdor_usd_idr.csv"

print(f"Input  : {RAW_PATH.relative_to(ROOT_DIR)}")
print(f"Output : {OUTPUT_PATH.relative_to(ROOT_DIR)}")

Input  : data/raw/Informasi_Kurs_Jisdor.xlsx
Output : data/processed/jisdor_usd_idr.csv


## 2. Deteksi Header

File BI punya beberapa baris judul sebelum tabel. Header dicari otomatis dari baris yang memuat kolom `Tanggal` dan `Kurs`.

In [2]:
excel_file = pd.ExcelFile(RAW_PATH)
if SHEET_NAME not in excel_file.sheet_names:
    raise ValueError(f"Sheet '{SHEET_NAME}' tidak ditemukan: {excel_file.sheet_names}")

header_scan = pd.read_excel(RAW_PATH, sheet_name=SHEET_NAME, header=None, nrows=50)


def is_header_row(row) -> bool:
    values = {str(v).strip() for v in row.dropna().tolist()}
    return {"Tanggal", "Kurs"}.issubset(values)


header_candidates = header_scan.index[header_scan.apply(is_header_row, axis=1)].tolist()
if not header_candidates:
    raise ValueError("Header dengan kolom 'Tanggal' dan 'Kurs' tidak ditemukan.")

HEADER_ROW = int(header_candidates[0])
print(f"Header terdeteksi di baris Excel ke-{HEADER_ROW + 1}.")

Header terdeteksi di baris Excel ke-5.


## 3. Memuat Tabel JISDOR

In [3]:
raw_df = pd.read_excel(RAW_PATH, sheet_name=SHEET_NAME, header=HEADER_ROW)
raw_df.columns = [str(c).strip() for c in raw_df.columns]

missing_columns = {"Tanggal", "Kurs"} - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {sorted(missing_columns)}")

print(f"Jumlah baris mentah: {len(raw_df):,}")
display(raw_df.head())

Jumlah baris mentah: 1,202


,NO,Tanggal,Kurs,Unnamed: 3
0,1,9/1/2026 12:00:00 AM,17727,NaN
1,2,8/31/2026 12:00:00 AM,17746,NaN
2,3,8/28/2026 12:00:00 AM,17703,NaN
3,4,8/27/2026 12:00:00 AM,17762,NaN
4,5,8/26/2026 12:00:00 AM,17717,NaN


## 4. Membentuk `date` dan `usd_idr`

Kolom `NO` dibuang. `Tanggal` diparse ke datetime (format sudah dicek konsisten di seluruh data), `Kurs` dipastikan numerik tanpa mengubah nilainya.

In [4]:
def parse_kurs(value):
    """Konversi Kurs ke float tanpa mengubah nilai ekonominya."""
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = re.sub(r"\s+", "", re.sub(r"(?i)rp", "", str(value).strip()))
    if re.fullmatch(r"\d{1,3}(?:\.\d{3})+(?:,\d+)?", text) or ("," in text and "." in text):
        text = text.replace(".", "").replace(",", ".")
    elif "," in text:
        text = text.replace(",", ".")
    return pd.to_numeric(text, errors="coerce")


working_df = (
    raw_df[["Tanggal", "Kurs"]]
    .rename(columns={"Tanggal": "date", "Kurs": "usd_idr"})
    .copy()
)
working_df["date"] = pd.to_datetime(
    working_df["date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce"
)
working_df["usd_idr"] = pd.to_numeric(working_df["usd_idr"].apply(parse_kurs), errors="coerce")

invalid_mask = working_df["date"].isna() | working_df["usd_idr"].isna()
clean_df = working_df.loc[~invalid_mask].copy()

outside_mask = (clean_df["date"] < START_DATE) | (clean_df["date"] > END_DATE)
outside_period_count = int(outside_mask.sum())

jisdor_df = (
    clean_df.loc[~outside_mask]
    .sort_values("date")
    .reset_index(drop=True)
)

if ((jisdor_df["usd_idr"] % 1) == 0).all():
    jisdor_df["usd_idr"] = jisdor_df["usd_idr"].astype("int64")

print(f"Baris tidak valid dibuang   : {int(invalid_mask.sum()):,}")
print(f"Baris di luar periode tugas : {outside_period_count:,}")
display(jisdor_df.head())

Baris tidak valid dibuang   : 0
Baris di luar periode tugas : 0


,date,usd_idr
0,2021-09-01,14284
1,2021-09-02,14281
2,2021-09-03,14261
3,2021-09-06,14239
4,2021-09-07,14195


## 5. Validasi Kualitas Data

In [5]:
duplicate_count = int(jisdor_df["date"].duplicated().sum())
missing_date_count = int(jisdor_df["date"].isna().sum())
missing_rate_count = int(jisdor_df["usd_idr"].isna().sum())

validation_summary = pd.DataFrame({
    "Pemeriksaan": [
        "Jumlah observasi", "Tanggal minimum", "Tanggal maksimum",
        "Duplicate date", "Missing date", "Missing usd_idr",
        "Data di luar periode (sebelum filter)",
    ],
    "Hasil": [
        f"{len(jisdor_df):,}",
        jisdor_df["date"].min().strftime("%Y-%m-%d"),
        jisdor_df["date"].max().strftime("%Y-%m-%d"),
        duplicate_count, missing_date_count, missing_rate_count,
        outside_period_count,
    ],
})
display(validation_summary)

if duplicate_count > 0:
    display(jisdor_df.loc[jisdor_df["date"].duplicated(keep=False)])
    raise ValueError("Ditemukan duplicate date.")

if missing_date_count > 0 or missing_rate_count > 0:
    raise ValueError("Masih terdapat missing value pada dataset final.")

if jisdor_df["date"].min() < START_DATE or jisdor_df["date"].max() > END_DATE:
    raise ValueError("Ada data di luar rentang periode tugas (2021-09-01 s.d. 2026-09-01).")

if jisdor_df["date"].min() != START_DATE or jisdor_df["date"].max() != END_DATE:
    print(
        "Catatan: observasi pertama/terakhir tidak persis di tanggal batas "
        f"(kemungkinan non-trading day): {jisdor_df['date'].min().date()} s.d. {jisdor_df['date'].max().date()}"
    )

print("Validasi selesai: data siap digunakan.")

,Pemeriksaan,Hasil
0,Jumlah observasi,"1,202"
1,Tanggal minimum,2021-09-01
2,Tanggal maksimum,2026-09-01
3,Duplicate date,0
4,Missing date,0
5,Missing usd_idr,0
6,Data di luar periode (sebelum filter),0


Validasi selesai: data siap digunakan.


## 6. Non-Trading Days

Gap tanggal hanya didokumentasikan, **tidak diisi/diinterpolasi**, non-trading days ditangani di tahap temporal alignment.

In [6]:
gap_days = jisdor_df["date"].diff().dt.days.dropna().astype(int)

gap_summary = (
    gap_days.value_counts()
    .sort_index()
    .rename_axis("selisih_hari")
    .reset_index(name="jumlah")
)
display(gap_summary)

,selisih_hari,jumlah
0,1,922
1,2,18
2,3,223
3,4,18
4,5,13
5,6,2
6,8,2
7,11,2
8,12,1


## 7. Export CSV

Disimpan sebagai `date` (`YYYY-MM-DD`) dan `usd_idr`. 

In [7]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
jisdor_df.to_csv(OUTPUT_PATH, index=False, date_format="%Y-%m-%d")

if not OUTPUT_PATH.exists():
    raise IOError("File output gagal dibuat.")

print(f"Tersimpan: {OUTPUT_PATH.relative_to(ROOT_DIR)}")
print(f"Jumlah observasi: {len(jisdor_df):,}")
display(jisdor_df.head())
display(jisdor_df.tail())

Tersimpan: data/processed/jisdor_usd_idr.csv
Jumlah observasi: 1,202


,date,usd_idr
0,2021-09-01,14284
1,2021-09-02,14281
2,2021-09-03,14261
3,2021-09-06,14239
4,2021-09-07,14195


,date,usd_idr
1197,2026-08-26,17717
1198,2026-08-27,17762
1199,2026-08-28,17703
1200,2026-08-31,17746
1201,2026-09-01,17727


## 8. Ringkasan

- **Sumber:** Bank Indonesia
- **Periode:** 2021-09-01 s.d. 2026-09-01
- **Kolom:** `date`, `usd_idr`
- **Non-trading days:** tidak diisi/diinterpolasi
- **Output:** `data/processed/jisdor_usd_idr.csv`